In [35]:
%pip install --upgrade pip
%pip install --upgrade numpy librosa soundfile ffmpeg-python
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cpu
%pip install --upgrade openai-whisper


Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import shutil
import subprocess

def resolve_ffmpeg():
    ffmpeg_path = shutil.which("ffmpeg")
    if ffmpeg_path:
        return ffmpeg_path

    machine_path = os.environ.get("Path", "")
    try:
        import winreg
        for hive, key in [
            (winreg.HKEY_LOCAL_MACHINE, r"SYSTEM\CurrentControlSet\Control\Session Manager\Environment"),
            (winreg.HKEY_CURRENT_USER, r"Environment"),
        ]:
            with winreg.OpenKey(hive, key) as reg_key:
                value, _ = winreg.QueryValueEx(reg_key, "Path")
                machine_path = f"{machine_path};{value}" if machine_path else value
    except Exception:
        pass

    os.environ["Path"] = machine_path
    return shutil.which("ffmpeg")

ffmpeg_exe = resolve_ffmpeg()
if ffmpeg_exe:
    result = subprocess.run([ffmpeg_exe, "-version"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    first_line = (result.stdout or result.stderr).splitlines()[0] if (result.stdout or result.stderr) else ""
    print("FFmpeg is installed and working.")
    if first_line:
        print(first_line)
    print(f"Using executable: {ffmpeg_exe}")
else:
    print("FFmpeg is installed on your system, but this notebook kernel cannot see it yet.")
    print("Restart the notebook kernel once, then rerun this cell.")

FFmpeg is installed and working.
ffmpeg version 8.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
Using executable: C:\Users\Shubhankar\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.1-full_build\bin\ffmpeg.EXE


In [37]:
import whisper
import numpy as np
import librosa
import soundfile as sf


In [38]:
print("Loading Whisper model...")

model = whisper.load_model("medium")

print("Model loaded successfully")

Loading Whisper model...
Model loaded successfully


In [39]:
from pathlib import Path

def load_audio(filepath):
    """Load an audio file path and resolve it robustly."""
    if not filepath:
        raise ValueError("No file path provided.")

    user_path = Path(filepath)
    candidate_paths = [user_path, Path.cwd() / user_path]

    if "project_dir" in globals():
        candidate_paths.append(Path(project_dir) / user_path)

    for candidate in candidate_paths:
        if candidate.exists():
            resolved = str(candidate.resolve())
            print(f"Using audio file: {resolved}")
            return resolved

    tried = "\n".join(str(p) for p in candidate_paths)
    raise FileNotFoundError(f"Audio file not found. Tried:\n{tried}")


In [40]:
def preprocess_audio(input_file):

    audio, sr = librosa.load(input_file, sr=16000)

    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio / peak

    output_file = "clean_audio.wav"
    sf.write(output_file, audio, sr)

    return output_file

In [41]:
def speech_to_text(audio_path):
    print("Transcribing audio...")

    # Use librosa audio array input to avoid depending on ffmpeg CLI availability.
    audio, _ = librosa.load(audio_path, sr=16000, mono=True)
    result = model.transcribe(audio, language="en")

    text = result["text"]
    return text

In [42]:
from pathlib import Path
import os

workspace_dir = Path(r"C:\Users\Shubhankar\Downloads\SIT\TYCS\ASR-3dForensic")
if workspace_dir.exists():
    os.chdir(workspace_dir)

candidate_names = ["recorded_audio.wav", "clean_audio.wav"]

search_dirs = [Path.cwd()]
if workspace_dir.exists():
    search_dirs.append(workspace_dir)
if "project_dir" in globals():
    candidate_project_dir = Path(project_dir)
    if candidate_project_dir.exists():
        search_dirs.append(candidate_project_dir)

search_dirs = list(dict.fromkeys(search_dirs))

audio_path = None
for name in candidate_names:
    for base in search_dirs:
        candidate = base / name
        if candidate.exists():
            audio_path = str(candidate)
            break
    if audio_path is not None:
        break

if audio_path is None:
    for base in search_dirs:
        for ext in ("*.wav", "*.mp3", "*.flac", "*.m4a"):
            matches = sorted(base.glob(ext))
            if matches:
                audio_path = str(matches[0])
                print(f"Using detected audio file: {audio_path}")
                break
        if audio_path is not None:
            break

if audio_path is None:
    searched = "\n".join(str(d) for d in search_dirs)
    raise FileNotFoundError(
        "No audio file found.\n"
        f"Current kernel folder: {Path.cwd()}\n"
        f"Searched folders:\n{searched}\n\n"
        "If your WAV is in your Windows project folder, switch this notebook to a local "
        "Python kernel in VS Code and rerun from Cell 1."
    )

audio_file = load_audio(audio_path)
clean_audio = preprocess_audio(audio_file)
transcription = speech_to_text(clean_audio)

print("Transcription:", transcription)


Using audio file: C:\Users\Shubhankar\Downloads\SIT\TYCS\ASR-3dForensic\recorded_audio.wav
Transcribing audio...
Transcription:  Old man with a wide jaw


In [43]:
def clean_text(text):
    text = text.lower()
    text = text.replace(",", "")
    text = text.replace(".", "")
    return text

cleaned_text = clean_text(transcription)

print("Cleaned text:", cleaned_text)

Cleaned text:  old man with a wide jaw


In [44]:
%pip install --upgrade typing_extensions pydantic pydantic-core
%pip install --upgrade spacy
%pip install --upgrade https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl

import importlib
import typing_extensions
importlib.reload(typing_extensions)

import spacy
nlp = spacy.load("en_core_web_sm")
print("spaCy ready:", spacy.__version__)

  Using cached pydantic_core-2.42.0-cp311-cp311-win_amd64.whl.metadata (6.7 kB)
Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 11.3 MB/s eta 0:00:02
     ------------- -------------------------- 4.5/12.8 MB 13.4 MB/s eta 0:00:01
     ---------------- ----------------------- 5.2/12.8 MB 9.7 MB/s eta 0:00:01
     ------------------ --------------------- 5.8/12.8 MB 8.2 MB/s eta 0:00:01
     -------------------- ------------------- 6.6/12.8 MB 6.7 MB/s eta 0:00:01
     ---------------------- ----------------- 7.1/12.8 MB 6.1 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 5.8 MB/s eta 0:00:01
     ----------------------- ---------------- 7.6/12.8 MB 5.8 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/12.8 MB 4.9 MB/s eta 0:00:01
     -------------------------------------- - 12.3/12.8 MB 6.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 6.0 MB/s  0:00:02
Note: you may need to restart the kernel to use updated pa

spaCy ready: 3.8.11


In [45]:
attribute_keywords = {
    "gender": [
        "a male face",
        "a female face"
    ],
    "approx_age": [
        "a child face",
        "a teenage face",
        "a young adult face",
        "a middle aged face",
        "an elderly face"
    ],
    "ethnicity_complexion": [
        "a light skin tone face",
        "a medium skin tone face",
        "a dark skin tone face",
        "an olive skin tone face"
    ],
    "skin_tone": [
        "fair skin",
        "medium skin",
        "dark skin",
        "olive skin"
    ],
    "facial_expression": [
        "a smiling face",
        "a neutral face",
        "a frowning face",
        "a surprised face",
        "an angry face"
    ],
    "clothing": [
        "casual clothing",
        "formal clothing"
    ],
    "face_shape": [
        "a face with an oval shape",
        "a face with a round shape",
        "a face with a square shape",
        "a face with a rectangular shape",
        "a face with a heart shape",
        "a face with a diamond shape"
    ],
    "jaw_strength": [
        "a face with a strong jawline",
        "a face with a soft jawline"
    ],
    "jaw_width": [
        "a face with a wide jaw",
        "a face with a narrow jaw"
    ],
    "chin_shape": [
        "a face with a pointed chin",
        "a face with a rounded chin",
        "a face with a receding chin",
        "a face with a double chin"
    ],
    "cheekbones": [
        "a face with high cheekbones",
        "a face with flat cheeks",
        "a face with full cheeks",
        "a face with hollow cheeks"
    ],
    "eye_size": [
        "a face with small eyes",
        "a face with medium sized eyes",
        "a face with large eyes"
    ],
    "eye_shape": [
        "a face with round eyes",
        "a face with almond shaped eyes",
        "a face with narrow eyes"
    ],
    "eye_spacing": [
        "a face with close set eyes",
        "a face with wide set eyes"
    ],
    "eye_tilt": [
        "a face with upward slanted eyes",
        "a face with downward slanted eyes",
        "a face with horizontally aligned eyes"
    ],
    "nose_length": [
        "a face with a short nose",
        "a face with a medium length nose",
        "a face with a long nose"
    ],
    "nose_width": [
        "a face with a narrow nose",
        "a face with a wide nose"
    ],
    "nose_bridge": [
        "a face with a straight nose bridge",
        "a face with a curved nose bridge",
        "a face with a flat nose bridge"
    ],
    "nose_tip": [
        "a face with a pointed nose tip",
        "a face with a round nose tip",
        "a face with a bulbous nose tip"
    ],
    "mouth_width": [
        "a face with a narrow mouth",
        "a face with a wide mouth"
    ],
    "mouth_corners": [
        "a face with upturned mouth corners",
        "a face with downturned mouth corners"
    ],
    "lip_ratio": [
        "a face with a larger upper lip",
        "a face with a larger lower lip",
        "a face with balanced lips"
    ],
    "ear_size": [
        "a face with small ears",
        "a face with large ears"
    ],
    "ear_position": [
        "a face with protruding ears",
        "a face with flat ears"
    ],
    "earlobes": [
        "a face with attached earlobes",
        "a face with detached earlobes"
    ],
    "forehead_height": [
        "a face with a high forehead",
        "a face with a medium forehead",
        "a face with a low forehead"
    ],
    "forehead_width": [
        "a face with a wide forehead",
        "a face with a narrow forehead"
    ],
    "hair_color": [
        "a person with black hair",
        "a person with brown hair",
        "a person with blonde hair",
        "a person with red hair",
        "a person with gray hair"
    ],
    "hair_length": [
        "a person with short hair",
        "a person with medium hair",
        "a person with long hair"
    ],
    "hair_texture": [
        "a person with straight hair",
        "a person with wavy hair",
        "a person with curly hair"
    ],
    "hair_density": [
        "a person with thick hair",
        "a person with normal hair",
        "a person with thin hair",
        "a person with sparse hair"
    ],
    "hair_parting": [
        "a person with a middle part hairstyle",
        "a person with a side part hairstyle"
    ],
    "hairline": [
        "a person with a receding hairline",
        "a person with a widow's peak hairline"
    ],
    "facial_hair": [
        "a clean shaven face",
        "a face with stubble",
        "a face with a mustache",
        "a face with a beard",
        "none"
    ],
    "beard_style": [
        "a face with a goatee beard",
        "a face with a full beard",
        "none"
    ],
    "beard_density": [
        "a face with a dense beard",
        "a face with a sparse beard",
        "a face with a patchy beard",
        "none"
    ],
    "eye_color": [
        "brown eyes",
        "blue eyes",
        "green eyes",
        "dark eyes",
        "black eyes",
        "grey eyes"
    ],
    "sideburns": [
        "none",
        "short",
        "long"
    ],
    "lip_color": [
        "a face with pale lips",
        "a face with pink lips",
        "a face with dark lips"
    ],
    "skin_texture": [
        "a face with smooth skin",
        "a face with rough skin",
        "a face with acne scars",
        "a face with visible pores"
    ],
    "wrinkles": [
        "a face with visible wrinkles",
        "a face with smooth skin"
    ],
    "freckles": [
        "a face with freckles",
        "a face without freckles"
    ],
    "moles": [
        "a face with moles",
        "a face without moles"
    ],
    "birthmarks": [
        "a face with birthmarks",
        "a face without birthmarks"
    ],
    "scars": ["yes", "no"],
    "tattoos": ["yes", "no"],
    "facial_injury": ["yes", "no"],
    "lazy_eye": ["yes", "no"],
    "crooked_nose": ["yes", "no"],
    "missing_tooth": ["yes", "no"],
    "piercings": ["yes", "no"]
}

In [46]:
import re

def _normalize_text_for_matching(value):
    value = (value or "").lower()
    value = value.replace("'", "")
    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value

def extract_attributes(text):
    safe_text = text or ""
    cleaned = clean_text(safe_text) if "clean_text" in globals() else safe_text.lower()
    normalized_text = _normalize_text_for_matching(cleaned)

    attributes = {key: None for key in attribute_keywords.keys()}

    # Phrase-first matching to support detailed multi-word categories.
    for attr, values in attribute_keywords.items():
        for phrase in values:
            norm_phrase = _normalize_text_for_matching(phrase)
            if norm_phrase and norm_phrase in normalized_text:
                attributes[attr] = phrase
                break

    # Token fallback matching for short descriptors (e.g., beard, freckles, wrinkles).
    if "nlp" in globals():
        doc = nlp(cleaned)
        token_set = {
            _normalize_text_for_matching(token.text)
            for token in doc
            if token.text and not token.is_space
        }
        token_set |= {
            _normalize_text_for_matching(token.lemma_)
            for token in doc
            if token.lemma_ and not token.is_space
        }

        stop_tokens = {
            "a", "an", "the", "with", "without", "face", "person", "of", "and", "to", "in"
        }

        for attr, values in attribute_keywords.items():
            if attributes[attr] is not None:
                continue
            for phrase in values:
                norm_phrase = _normalize_text_for_matching(phrase)
                phrase_tokens = [
                    t for t in norm_phrase.split()
                    if t and t not in stop_tokens
                ]
                if phrase_tokens and all(token in token_set for token in phrase_tokens):
                    attributes[attr] = phrase
                    break

    return attributes

if "cleaned_text" in globals():
    attributes = extract_attributes(cleaned_text)
    print("Extracted Attributes:", attributes)
else:
    print("cleaned_text not found yet. Run transcription and cleaning cells first.")

Extracted Attributes: {'gender': None, 'approx_age': None, 'ethnicity_complexion': None, 'skin_tone': None, 'facial_expression': None, 'clothing': None, 'face_shape': None, 'jaw_strength': None, 'jaw_width': 'a face with a wide jaw', 'chin_shape': None, 'cheekbones': None, 'eye_size': None, 'eye_shape': None, 'eye_spacing': None, 'eye_tilt': None, 'nose_length': None, 'nose_width': None, 'nose_bridge': None, 'nose_tip': None, 'mouth_width': None, 'mouth_corners': None, 'lip_ratio': None, 'ear_size': None, 'ear_position': None, 'earlobes': None, 'forehead_height': None, 'forehead_width': None, 'hair_color': None, 'hair_length': None, 'hair_texture': None, 'hair_density': None, 'hair_parting': None, 'hairline': None, 'facial_hair': None, 'beard_style': None, 'beard_density': None, 'eye_color': None, 'sideburns': None, 'lip_color': None, 'skin_texture': None, 'wrinkles': None, 'freckles': None, 'moles': None, 'birthmarks': None, 'scars': None, 'tattoos': None, 'facial_injury': None, 'lazy

In [47]:
audio_file = load_audio("recorded_audio.wav")
clean_audio = preprocess_audio(audio_file)
transcription = speech_to_text(clean_audio)

cleaned_text = clean_text(transcription)
print("Cleaned:", cleaned_text)

attributes = extract_attributes(cleaned_text)
print("Final Attributes:", attributes)

Using audio file: C:\Users\Shubhankar\Downloads\SIT\TYCS\ASR-3dForensic\recorded_audio.wav
Transcribing audio...
Cleaned:  old man with a wide jaw
Final Attributes: {'gender': None, 'approx_age': None, 'ethnicity_complexion': None, 'skin_tone': None, 'facial_expression': None, 'clothing': None, 'face_shape': None, 'jaw_strength': None, 'jaw_width': 'a face with a wide jaw', 'chin_shape': None, 'cheekbones': None, 'eye_size': None, 'eye_shape': None, 'eye_spacing': None, 'eye_tilt': None, 'nose_length': None, 'nose_width': None, 'nose_bridge': None, 'nose_tip': None, 'mouth_width': None, 'mouth_corners': None, 'lip_ratio': None, 'ear_size': None, 'ear_position': None, 'earlobes': None, 'forehead_height': None, 'forehead_width': None, 'hair_color': None, 'hair_length': None, 'hair_texture': None, 'hair_density': None, 'hair_parting': None, 'hairline': None, 'facial_hair': None, 'beard_style': None, 'beard_density': None, 'eye_color': None, 'sideburns': None, 'lip_color': None, 'skin_text

In [50]:
import json

def present(value, default='Not available'):
    return value if value is not None else default

summary = {
    'Input Audio File': present(globals().get('audio_file')),
    'Preprocessed Audio File': present(globals().get('clean_audio')),
    'Raw Transcription': present(globals().get('transcription')),
    'Cleaned Text': present(globals().get('cleaned_text')),
    'Extracted Attributes': present(globals().get('attributes'), {}),
    'spaCy Loaded': 'Yes' if 'nlp' in globals() else 'No',
    'Whisper Model Loaded': 'Yes' if 'model' in globals() else 'No'
}

raw_attributes = summary.get('Extracted Attributes', {})
if isinstance(raw_attributes, dict):
    compact_attributes = {
        k: v for k, v in raw_attributes.items()
        if v is not None and str(v).strip() != ''
    }
else:
    compact_attributes = {}

print('Summary')
print('=' * 72)
for key, value in summary.items():
    if key == 'Extracted Attributes':
        print('Extracted Attributes (Non-Null Only):')
        if compact_attributes:
            print(json.dumps(compact_attributes, indent=2))
            print(f'Matched Attributes: {len(compact_attributes)} / {len(raw_attributes)}')
        else:
            print('{}')
            print('Matched Attributes: 0')
    else:
        print(f'{key}: {value}')

print('\n' + '-' * 72)
print('Pipeline Status')
print('-' * 72)
print('1) ASR Pipeline: Completed')
print('2) Text Cleaning: Completed')
print('3) NLP Extraction (spaCy): Completed')
print('=' * 72)

Summary
Input Audio File: C:\Users\Shubhankar\Downloads\SIT\TYCS\ASR-3dForensic\recorded_audio.wav
Preprocessed Audio File: clean_audio.wav
Raw Transcription:  Old man with a wide jaw
Cleaned Text:  old man with a wide jaw
Extracted Attributes (Non-Null Only):
{
  "jaw_width": "a face with a wide jaw"
}
Matched Attributes: 1 / 51
spaCy Loaded: Yes
Whisper Model Loaded: Yes

------------------------------------------------------------------------
Pipeline Status
------------------------------------------------------------------------
1) ASR Pipeline: Completed
2) Text Cleaning: Completed
3) NLP Extraction (spaCy): Completed
